# VaxiMère-QA-CG — Entraînement du petit modèle de test

Deux phases, sur GPU T4 :
1. **Encodeur léger** (backbone multilingue `cis-lmu/glot500-base`) — rapide,
   produit accuracy / F1 / matrice de confusion.
2. **Petit LLM + LoRA** (`google/gemma-2-2b-it`) — reproduit la cible finale
   (Gemma 3 + LoRA) en instruction tuning.

Le split train/val/test est fait **au niveau de la question maîtresse** pour
éviter que la traduction d'une question ne fuite entre train et test.

In [ ]:
# 0) Dépendances
!pip install -q datasets transformers pandas accelerate sentencepiece \
  peft trl bitsandbytes scikit-learn huggingface_hub

In [ ]:
# 1) Récupération du code (clone + données déjà générées)
import os, subprocess
from pathlib import Path

if not Path("/content/AIMS-Capstone").exists():
    subprocess.run(["git", "clone", "--branch", "arena/01a03c0c-aims-capstone",
                    "https://github.com/maick-code/AIMS-Capstone.git",
                    "/content/AIMS-Capstone"], check=True)
os.chdir("/content/AIMS-Capstone")

# récupère le dernier code + les données générées
subprocess.run(["git", "fetch", "origin", "arena/01a03c0c-aims-capstone"], check=True)
subprocess.run(["git", "checkout", "origin/arena/01a03c0c-aims-capstone", "--",
                "vaximere/", "data/", "requirements.txt"], check=True)

print("data ok:", Path("data/vaximere_qa_cg_train.jsonl").exists())
print("training ok:", Path("vaximere/training").exists())

In [ ]:
# 2) Vérif GPU
!nvidia-smi -L
import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 3) Validation du split (sans GPU, rapide)
!python selftest_split.py

In [ ]:
# 4) PHASE 1 — Classifieur encodeur (rapide : fp16 + sauvegarde finale uniquement)
#     Backbone : glot500-base (couvre fra + lingala + kikongo/kituba).
#     lr=5e-5, 8 époques : mieux adapté à un entraînement aussi court.
!python vaximere/training/train_encoder.py \
    --jsonl data/vaximere_qa_cg_train.jsonl \
    --model-name cis-lmu/glot500-base \
    --epochs 8 --batch-size 32 --lr 5e-5 \
    --out outputs/encoder


In [ ]:
# 5) PHASE 2 — Petit LLM + LoRA (QLoRA 4-bit, ~10-20 min sur T4)
#     Modèle par défaut : Qwen/Qwen2.5-0.5B-Instruct (NON gated, Apache-2.0).
#     ⚠️ Les modèles Gemma/Llama sont « gated » : il faut d'abord accepter leur
#     licence sur huggingface.co (page du modèle) pour pouvoir les télécharger.
!python vaximere/training/train_decoder.py \
    --jsonl data/vaximere_qa_cg_train.jsonl \
    --model-name Qwen/Qwen2.5-0.5B-Instruct \
    --epochs 3 --batch-size 2 \
    --out outputs/decoder_lora


In [ ]:
# 6) Pousser le modèle encodeur sur le Hub (token masqué) — SANS re-entraîner
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Token HF (write) : ")

!python vaximere/training/train_encoder.py \
    --jsonl data/vaximere_qa_cg_train.jsonl \
    --model-name cis-lmu/glot500-base \
    --out outputs/encoder \
    --push-only --hub-model-id Semence/vaximere-intent-glot500


In [ ]:
# 7) Inférence rapide avec le modèle encodeur entraîné
from pathlib import Path

model_path = Path("outputs/encoder/model")
if not model_path.exists():
    print("❌ Le modèle n'existe pas : lancez la cellule 4 (Phase 1) et vérifiez "
          "qu'elle se termine SANS erreur (regardez les logs au-dessus).")
else:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    import torch

    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForSequenceClassification.from_pretrained(str(model_path))
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    exemples = [
        "Mon bébé a de la fièvre après le vaccin, est-ce normal ?",
        "On dit que le vaccin rend les enfants stériles, est-ce vrai ?",
        "Mwana na ngai azali na fièvre nsima ya vaccin, ezali malamu ?",
        "Sambu na nki kupesa mwana na mono vaccine ?",
    ]
    for t in exemples:
        enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=128).to(model.device)
        with torch.no_grad():
            logits = model(**enc).logits
        pred = model.config.id2label[int(logits.argmax(-1))]
        print(f"{t[:60]:<62} -> {pred}")
